## Re-fitting Candidate Metal Lines with Boostrapping and Autocorrelation Models

This script re-fits all putative emission and absorption lines, this time with robust error estimation taking into account autocorrelation with fit_mc.

After re-fitting, advanced flagging is performed to check for possible systematics

In [ ]:
import numpy as np
import astropy.table as aptb
from astropy.io import fits

from tangelo import spectroscopy as tgspec
from tangelo import fitting as tgfit
from tangelo import catalogue_operations as tgcat
from tangelo import constants as tgconst
from tangelo import io as tgio
from tangelo.quality_control import flag_fit_result

In [ ]:
# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? R21 or 2fwhm (apertures)
SPEC_TYPE   = "weight_skysub" # Which subtype of spectrum to use (e.g. 'Nfwhm_opt' for APER, 'weight_skysub' for R21)

tabfile = f"./megatables/fit_catalog_qc_cpts_stack_zeldamcmc_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

writename = f"./megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_{SPEC_TYPE}.fits"

In [ ]:
# Update all flag columns to dtype U10
for col in megatable.colnames:
    if 'FLAG_' in col:
        megatable[col] = megatable[col].astype('U10')

In [ ]:
# Go through each source and refit each putative detection of emission or absorption lines

import importlib

importlib.reload(tgspec)
importlib.reload(tgfit)
importlib.reload(tgcat)

for i, row in enumerate(megatable):
    
    iden = row['iden']
    cluster = row['CLUSTER']
    idfrom = row['idfrom']
    
    # Where is this source in the megatab
    megatab_index = i
    
    print(f"\n{'='*40}\nProcessing source {i+1}/{len(megatable)}: {cluster} {iden} (idfrom={idfrom})\n{'='*40}")

    # Find all the statistically significant candidate lines for this source by searching columns
    candidate_lines = []
    for col in megatable.colnames:
        if 'SNR_' in col and np.abs(megatable[i][col]) > 3.0:  # Using 3.0 as a threshold to be inclusive of potential candidates
            linename = col.split('_')[1]
            print(f"Found candidate {linename} line with SNR = {megatable[col][i]}")
            candidate_lines.append(linename)
    
    # Check for doublets: if we find a secondary line, replace it with the primary, unless 
    # the primary doesn't appear in the columns of megatab
    # (refit_other_line will automatically fit both components when given the primary)
    lines_to_fit = []
    for line in candidate_lines:
        # Check if this is a secondary line
        if np.any(tgconst.slines == line):
            # Find the corresponding primary line
            primary_line = tgconst.flines[np.where(tgconst.slines == line)[0][0]]
            if primary_line not in lines_to_fit and f"SNR_{primary_line}" in megatable.colnames:
                lines_to_fit.append(primary_line)
                print(f"{line} is secondary - will fit primary {primary_line} instead")
        else:
            # Either a primary line, not a doublet, or the primary line isn't in the table columns - fit as is
            if line not in lines_to_fit:
                lines_to_fit.append(line)

    # If no candidate lines, skip
    if len(lines_to_fit) == 0:
        print("No candidate lines found, skipping...")
        continue

    # If lines found, load the spectrum for this source
    spectab = tgio.load_spec(cluster, iden, idfrom, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE)
    if spectab is None:
        raise ValueError("Spectrum could not be loaded -- check files")  # Raise an error rather than moving to the next source

    # Get wavelength, flux, err arrays
    flux = spectab['spec']
    wave = spectab['wave']
    err = spectab['spec_err']

    # Refit each candidate line
    line_fit_results = {}
    for line in lines_to_fit:

        print("\n")
    
        fit_results_dict = tgfit.refit_other_line(
            wave, flux, err, row,
            line_name=line,
            width=50,
            bootstrap_params={
                'niter': 1000,
                'max_nfev': 500,
                'errfunc': 'stddev_7',
                'chisq_thresh': np.inf,
                'sig_clip': 20.0,
                'autocorrelation': True,
                'baseline_order': 1
            },
        )
        
        if 'param_dict' not in fit_results_dict:
            print(f"Fit failed for {line}. Moving to next line...")
            continue

        snr = fit_results_dict['param_dict']['FLUX'] / fit_results_dict['error_dict']['FLUX']
        snr2 = fit_results_dict['param_dict'].get('FLUX2', 0) / fit_results_dict['error_dict'].get('FLUX2', 1)  # Avoid division by zero if FLUX2 not present
        total_snr = np.sqrt(snr**2 + snr2**2)  # Combine SNRs in quadrature if both components are present

        doublet_name = line + (('+' + tgconst.doublets[line][1]) if line in tgconst.doublets else '')

        if np.abs(total_snr) < 3.0:
            print(f"\nRefitted line {doublet_name} has SNR < 3 ({total_snr:.2f})")
        elif np.abs(total_snr) >= 3.0:
            print(f"\nRefitted line {doublet_name} has SNR >= 3 ({total_snr:.2f})")

        # Flagging
        flags = ['','']
        if total_snr > 3.0:
            # Don't waste time flagging lines that aren't significant
            flags = flag_fit_result(fit_results_dict, line)
        fit_results_dict['flags'] = flags
        
        print(f"\nFlags applied: {fit_results_dict['flags']}")

        line_fit_results[line] = fit_results_dict
        
    # Pass results to megatab
    tgcat.insert_fit_results(megatable, cluster, iden, None, line_fit_results, replace_stacked=False)
    
    # Save table in case of crash
    megatable.write(writename, overwrite=True)

    print(f"\nSummary for {cluster} {iden}:")
    for line in line_fit_results.keys():
        best_params = line_fit_results[line]['param_dict']
        best_errs = line_fit_results[line]['error_dict']
        snr = best_params['FLUX'] / best_errs['FLUX']
        
        # Print the final SNR of the line (or doublet)
        print(f"{line} SNR: {snr:.2f}; FLAGS: {line_fit_results[line]['flags'][0]}")
        if 'FLUX2' in best_params:
            partner_name = tgconst.doublets[line][1]
            snr2 = best_params['FLUX2'] / best_errs['FLUX2']
            print(f"{partner_name} SNR: {snr2:.2f}; FLAGS: {line_fit_results[line]['flags'][1]}")
            
        

In [ ]:
megatable.write(writename, overwrite=True)

In [ ]:
# How many true emitters and absorbers are left?

trueemitters = []
trueabsorbers = []

# Extract only SNR column names
snr_lines = [n.split('_')[1] for n in megatable.colnames if 'SNR_' in n]

# Find true emitters (any SNR > 3.0)
trueemitters = []
for row in megatable:
    if any(row[f'SNR_{line}'] > 3.0  and row[f'FLAG_{line}'] == '' for line in snr_lines):
        trueemitters.append(row)
        
# Find true absorbers (any SNR < -3.0)
trueabsorbers = []
for row in megatable:
    if any(row[f'SNR_{line}'] < -3.0 and row[f'FLAG_{line}'] == '' for line in snr_lines):
        trueabsorbers.append(row)

In [ ]:
# Print summary
print(f"Total sources: {len(megatable)}")
print(f"True emitters (SNR > 3): {len(trueemitters)}")
print(f"True absorbers (SNR < -3): {len(trueabsorbers)}")